In [152]:
import pandas as pd
import geopandas as gpd
import maup

maup.progress.enabled = True

pd.options.mode.chained_assignment = None
pd.set_option('display.max_columns', None)

# Because we can't store the data in the Repository, I need to get out 
# of the Repo and back to my Laptop where I have it saved.
from pathlib import Path
BASE_DIR = Path().resolve().parent.parent

In [177]:

# Read in the shapefiles that we need - block and precinct.
blocks_gdf = gpd.read_file(BASE_DIR / "nc_block" / "tl_2020_37_tabblock20.shp") # Census Blocks
blocks_gdf = blocks_gdf.loc[blocks_gdf["COUNTYFP20"] == "101"] # Filter to just Johnston County

# Precinct data from: https://www.nconemap.gov/datasets/nconemap::voting-precincts/about
precincts_23 = gpd.read_file(BASE_DIR/ "Voting_Precincts" / "Voting_Precincts.shp") # Precinct Level
precincts_23 = precincts_23.loc[precincts_23["county_nam"] == "JOHNSTON"] # Get just precincts in Johnston County

# Election Data (For the Town Council election on 11/7 At-Large Member Candidates)
GIT_DIR = Path().resolve().parent
smithfield_tc = pd.read_csv(GIT_DIR / "Election_Data_Sheets" / "csv" / "results_pct_20231107.csv")
smithfield_tc = smithfield_tc.loc[smithfield_tc['Contest Name'] == 'TOWN OF SMITHFIELD TOWN COUNCIL MEMBERS AT-LARGE']
smithfield_tc = smithfield_tc.loc[smithfield_tc["Real Precinct"] == "Y"] 
smithfield_tc = smithfield_tc[["Precinct", "Choice", "Total Votes"]] # Precinct Level election results

# Demographic Data
demographic_df = gpd.read_file(BASE_DIR / "nc_demographic_data_2020" / "DECENNIALPL2020.P1-Data.csv") # Census Blocks


In [178]:
# Now, let's merge the demographic data with the Block precinct data
# We notice that we can match on GEO_ID, but demographic_df has more info than blocks_df. Let's cut the first 9 letters.
demographic_df["GEO_ID"] = demographic_df["GEO_ID"].str[9:]



In [179]:
# There's also a lot of columns that we don't need/can rename, so let's do that!
demographic_df = demographic_df[["GEO_ID", 'P1_001N', 'P1_002N', 'P1_003N', "P1_004N", "P1_005N", "P1_006N", "P1_007N", "P1_008N"]]
blocks_gdf['GEO_ID'] = blocks_gdf['GEOID20']
blocks_gdf = blocks_gdf[["GEO_ID", "POP20", "geometry"]]

# Now let's do the merge for block data
blocks_gdf = blocks_gdf.merge(demographic_df, on = ["GEO_ID"], how = "left")

In [180]:
# We need to pivot the Smithfield precincts so that each Precinct is one row (and the candidates are columns with number of votes in the row)
# We'll do this using pandas pivot.
smithfield_tc = smithfield_tc.pivot(index='Precinct', columns = 'Choice', values = 'Total Votes')
smithfield_tc = smithfield_tc.rename_axis(None, axis=1).reset_index()

In [183]:
# Now, let's do the same for precincts!

# There's also a lot of columns that we don't need/can rename, so let's do that!
precincts_23['Precinct'] = precincts_23['prec_id']
precincts_23 = precincts_23[["Precinct", 'geometry']]

# Now let's do the merge for block data
smithfield_precincts = precincts_23.merge(smithfield_tc, on = ["Precinct"], how = "left")

# During the merge, it changed it to a pandas dataframe. Let's change it back to a geodataframe.
smithfield_precincts = gpd.GeoDataFrame(smithfield_precincts, geometry="geometry", crs=blocks_gdf.crs)
# Drop the NaN values (for precincts not in Smithfield)
smithfield_precincts = smithfield_precincts.dropna()

In [184]:
smithfield_precincts

,Precinct,geometry,Doris Louise Wallace,Felicia C. Baxter,John A. Dunn,Roger A. Wood,Stephen Rabil,Stuart Ashby Lee,Write-In (Miscellaneous)
12,PR24,"POLYGON ((669326.41443 197828.29236, 669354.03...",4.0,4.0,5.0,5.0,2.0,2.0,0.0
20,PR26,"POLYGON ((665455.96583 190896.28386, 665460.98...",181.0,94.0,38.0,52.0,70.0,80.0,7.0
21,PR27A,"POLYGON ((669723.88483 195655.85396, 669703.51...",89.0,54.0,145.0,144.0,140.0,56.0,2.0
22,PR27B,"POLYGON ((664975.18603 196644.70756, 664881.41...",61.0,20.0,78.0,99.0,80.0,26.0,0.0
23,PR28,"POLYGON ((665549.24283 191118.51686, 665547.81...",132.0,66.0,348.0,352.0,331.0,131.0,1.0


In [185]:
blocks_gdf

,GEO_ID,POP20,geometry,P1_001N,P1_002N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N
0,371010401021015,11,"POLYGON ((-78.20378 35.6411, -78.20286 35.6414...",11,9,5,0,0,0,0,4
1,371010402073010,56,"POLYGON ((-78.37494 35.67086, -78.37474 35.670...",56,25,16,2,0,7,0,0
2,371010414022005,0,"POLYGON ((-78.56592 35.38798, -78.5659 35.3881...",0,0,0,0,0,0,0,0
3,371010414022029,11,"POLYGON ((-78.55782 35.38467, -78.55761 35.384...",11,9,6,0,0,3,0,0
4,371010411102011,14,"POLYGON ((-78.50413 35.62917, -78.50395 35.629...",14,11,5,1,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...
3742,371010415053005,466,"POLYGON ((-78.53059 35.43605, -78.53056 35.436...",466,452,388,39,3,0,0,22
3743,371010402051004,49,"POLYGON ((-78.36056 35.752, -78.36052 35.75208...",49,44,27,2,0,0,0,15
3744,371010413013007,27,"POLYGON ((-78.28957 35.40063, -78.28956 35.400...",27,24,20,0,1,0,0,3
3745,371010415102004,0,"POLYGON ((-78.5965 35.46509, -78.5963 35.46526...",0,0,0,0,0,0,0,0


In [188]:
precincts_23 = gpd.GeoDataFrame(precincts_23, geometry="geometry", crs=blocks_gdf.crs)


In [ ]:
# We now need to agregate the block data up to the precinct data
pop_cols = ['P1_001N', 'P1_002N', 'P1_003N', "P1_004N", "P1_005N", "P1_006N", "P1_007N", "P1_008N"]

# How blocks are inputted into the precinct shapefile dataframe
blocks_to_precincts_assignment = maup.assign(blocks_gdf.geometry, precincts_23.geometry)

# We need to get all the data types of P1 to ints
for item in pop_cols:
    blocks_gdf[item] = pd.to_numeric(blocks_gdf[item])
smithfield_precincts[pop_cols] = blocks_gdf[pop_cols].groupby(blocks_to_precincts_assignment).sum()
# This merge isn't working - that's a later issue for me!

100%|██████████| 36/36 [00:00<00:00, 4408.48it/s]
/opt/anaconda3/envs/VoteKit/lib/python3.13/site-packages/maup/intersections.py:49: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df = df[df.area > area_cutoff].reset_index(drop=True)
/opt/anaconda3/envs/VoteKit/lib/python3.13/site-packages/maup/intersections.py:50: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geometries = geometries[geometries.area > area_cutoff]
/opt/anaconda3/envs/VoteKit/lib/python3.13/site-packages/maup/assign.py:46: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  return assign_to_max(intersections(sources, target

In [191]:
missing = smithfield_precincts[smithfield_precincts[pop_cols].isna().any(axis=1)]
print(missing["Precinct"])

12     PR24
20     PR26
21    PR27A
22    PR27B
23     PR28
Name: Precinct, dtype: object


In [190]:
smithfield_precincts

,Precinct,geometry,Doris Louise Wallace,Felicia C. Baxter,John A. Dunn,Roger A. Wood,Stephen Rabil,Stuart Ashby Lee,Write-In (Miscellaneous),P1_001N,P1_002N,P1_003N,P1_004N,P1_005N,P1_006N,P1_007N,P1_008N
12,PR24,"POLYGON ((669326.41443 197828.29236, 669354.03...",4.0,4.0,5.0,5.0,2.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,PR26,"POLYGON ((665455.96583 190896.28386, 665460.98...",181.0,94.0,38.0,52.0,70.0,80.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,PR27A,"POLYGON ((669723.88483 195655.85396, 669703.51...",89.0,54.0,145.0,144.0,140.0,56.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,PR27B,"POLYGON ((664975.18603 196644.70756, 664881.41...",61.0,20.0,78.0,99.0,80.0,26.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,PR28,"POLYGON ((665549.24283 191118.51686, 665547.81...",132.0,66.0,348.0,352.0,331.0,131.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
